**Java 24+:** Spark/Hadoop need Java 17 (older JDKs work too). Run the cell below first — it sets `JAVA_HOME` and `YELP_JAVA17_HOME` from `mise` if installed — then **restart the kernel** and run all cells from the top.

In [1]:
# Run this first if you use Java 25+. Then restart kernel and run all.
import os, subprocess
path = ""
try:
    r = subprocess.run(["mise", "where", "java@17"], capture_output=True, text=True, timeout=5)
    path = (r.stdout or "").strip()
except Exception:
    pass
if path and os.path.exists(os.path.join(path, "bin", "java")):
    os.environ["YELP_JAVA17_HOME"] = path
    os.environ["JAVA_HOME"] = path
    print("Set JAVA_HOME and YELP_JAVA17_HOME to", path, "-- restart the kernel and run all cells.")
else:
    print("Install Java 17 first: run in terminal:  mise install java@17")

Set JAVA_HOME and YELP_JAVA17_HOME to /home/dash/.local/share/mise/installs/java/17.0.2 -- restart the kernel and run all cells.


In [2]:
from src.constants import PROCESSED_DIR
from src.preprocessing import (
    clean_all,
    flatten_all,
    load_all_raw,
    preprocess_all,
    reduce_all,
    transform_all,
    write_processed,
)
from src.spark.session import create_spark_session

In [3]:
# Optional: set *before* create_spark_session. After changing, run spark.stop() then create_spark_session again
# (kernel restart not required). Then re-run pipeline cells — DataFrames from the old session are invalid.
import os
# os.environ["SPARK_LOCAL_DIRS"] = "/var/tmp/spark-scratch"  # only if project/home disk is also quota-limited (default is artifacts/spark_local_scratch)
# os.environ["SPARK_MAX_CORES"] = "1"  # local[1] — lowest CPU load
# os.environ["SPARK_DRIVER_MEMORY"] = "768m"
# os.environ["SPARK_SQL_SHUFFLE_PARTITIONS"] = "4"
# os.environ["YELP_WRITE_PARTITIONS"] = "32"  # more Parquet shards → smaller per-task sorts

In [4]:
spark = create_spark_session("run_preprocessing")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 14:27:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/29 14:27:37 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


**Staged pipeline:** Run the cells below one by one. Each stage logs progress per dataset and a heartbeat every 60s so you can see it is still running.

**Resource limits:** `create_spark_session` defaults to `local[2]`, 1g driver memory, bounded shuffle partitions, and **`artifacts/spark_local_scratch`** for Spark shuffle/spill (not `/tmp`). Override **`SPARK_LOCAL_DIRS`** only if that path is still on a small quota. For Parquet writes, set `YELP_WRITE_PARTITIONS`. Launching Jupyter with `nice -n 10` can also keep the desktop more responsive under load.

In [5]:
# Stage 1: Load raw data (progress logged per dataset; heartbeat every 60s during long steps)
raw = load_all_raw(spark)

[15:27:41] Stage 1: Loading raw data (6 datasets)
[15:27:41]   [1/6] business: loading...


[15:27:44]   [1/6] business: done (150346 rows, 3.2s)
[15:27:44]   [2/6] review: loading...


[15:27:52]   [2/6] review: done (6990280 rows, 8.4s)
[15:27:52]   [3/6] user: loading...


[15:27:57]   [3/6] user: done (1987897 rows, 4.6s)
[15:27:57]   [4/6] checkin: loading...


[15:27:57]   [4/6] checkin: done (131930 rows, 0.5s)
[15:27:57]   [5/6] tip: loading...
[15:27:58]   [5/6] tip: done (908915 rows, 0.6s)
[15:27:58]   [6/6] photo: loading...
[15:27:58]   [6/6] photo: done (200100 rows, 0.2s)
[15:27:58] Stage 1: finished.


In [6]:
# Stage 2: Clean (nulls, duplicates)
cleaned = clean_all(spark, raw)

[15:27:58] Stage 2: Cleaning (6 datasets)
[15:27:58]   [1/6] business: cleaning...


[15:28:02]   [1/6] business: done (150346 rows, 4.0s)
[15:28:02]   [2/6] review: cleaning...


[15:29:02]   ... still running: clean review


[15:29:17]   [2/6] review: done (6990280 rows, 75.3s)
[15:29:17]   [3/6] user: cleaning...


[15:30:17]   ... still running: clean user


[15:30:53]   [3/6] user: done (1987897 rows, 95.5s)
[15:30:53]   [4/6] checkin: cleaning...


[15:30:55]   [4/6] checkin: done (131930 rows, 1.8s)
[15:30:55]   [5/6] tip: cleaning...


[15:30:58]   [5/6] tip: done (908836 rows, 3.0s)
[15:30:58]   [6/6] photo: cleaning...


[15:30:58]   [6/6] photo: done (200098 rows, 0.3s)
[15:30:58] Stage 2: finished.


In [7]:
# Stage 3: Flatten nested columns (attributes, hours, friends, elite, checkin date)
flattened = flatten_all(cleaned)

[15:30:58] Stage 3: Flattening (6 datasets)
[15:30:58]   [1/6] business: flattening...
[15:30:59]   [1/6] business: done (150346 rows, 1.0s)
[15:30:59]   [2/6] review: flattening...


[15:31:14]   [2/6] review: done (6990280 rows, 14.6s)
[15:31:14]   [3/6] user: flattening...


[15:31:19]   [3/6] user: done (1987897 rows, 5.0s)
[15:31:19]   [4/6] checkin: flattening...


[15:31:20]   [4/6] checkin: done (131930 rows, 1.6s)
[15:31:20]   [5/6] tip: flattening...


[15:31:22]   [5/6] tip: done (908836 rows, 1.6s)
[15:31:22]   [6/6] photo: flattening...


[15:31:22]   [6/6] photo: done (200098 rows, 0.2s)
[15:31:22] Stage 3: finished.


In [8]:
# Stage 4: Transform (scale, parse dates, drop redundant cols)
transformed = transform_all(spark, flattened)

[15:31:22] Stage 4: Transforming (6 datasets)
[15:31:22]   [1/6] business: transforming...


[15:31:27]   [1/6] business: done (150346 rows, 4.8s)
[15:31:27]   [2/6] review: transforming...


[15:32:27]   ... still running: transform review


[15:32:48]   [2/6] review: done (6990280 rows, 80.7s)
[15:32:48]   [3/6] user: transforming...


[15:33:48]   ... still running: transform user


[15:34:39]   [3/6] user: done (1987897 rows, 111.5s)
[15:34:39]   [4/6] checkin: transforming...


[15:34:44]   [4/6] checkin: done (131930 rows, 4.4s)
[15:34:44]   [5/6] tip: transforming...


[15:34:47]   [5/6] tip: done (908836 rows, 3.5s)
[15:34:47]   [6/6] photo: transforming...


[15:34:47]   [6/6] photo: done (200098 rows, 0.3s)
[15:34:47] Stage 4: finished.


In [9]:
# Stage 5: Reduce (optional sampling) and finish
processed = reduce_all(spark, transformed)

[15:34:47] Stage 5: Reduce (6 datasets)
[15:34:47]   [1/6] business: reduce...
[15:34:48]   [1/6] business: done (150346 rows, 0.3s)
[15:34:48]   [2/6] review: reduce...


[15:35:09]   [2/6] review: done (6990280 rows, 20.8s)
[15:35:09]   [3/6] user: reduce...


[15:35:13]   [3/6] user: done (1987897 rows, 4.0s)
[15:35:13]   [4/6] checkin: reduce...


[15:35:14]   [4/6] checkin: done (131930 rows, 1.6s)
[15:35:14]   [5/6] tip: reduce...


[15:35:16]   [5/6] tip: done (908836 rows, 1.7s)
[15:35:16]   [6/6] photo: reduce...


[15:35:16]   [6/6] photo: done (200098 rows, 0.3s)
[15:35:16] Stage 5: finished.


In [10]:
# Write processed DataFrames to Parquet (progress logged per dataset)
write_processed(processed)

[15:35:16] Writing Parquet (6 datasets) to /home/dash/University/term-6/big-data/yelp-spark-project/artifacts/processed (16 partitions each)
[15:35:16]   [1/6] business: writing...


26/03/29 14:35:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[15:35:24]   [1/6] business: done (7.5s)
[15:35:24]   [2/6] review: writing...


[15:42:37]   [2/6] review: done (433.2s)
[15:42:37]   [3/6] user: writing...


[15:42:57]   [3/6] user: done (20.6s)
[15:42:58]   [4/6] checkin: writing...


[15:43:02]   [4/6] checkin: done (4.3s)
[15:43:02]   [5/6] tip: writing...


[15:43:07]   [5/6] tip: done (5.7s)
[15:43:07]   [6/6] photo: writing...


[15:43:09]   [6/6] photo: done (1.1s)
[15:43:09] Writing: finished.
